# Bab 6: Evaluasi dan Pemilihan Model

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab06/akurasi_menipu.py": "import numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import confusion_matrix\nfrom data_transaksi import buat_transaksi\n\nX, y, hari = buat_transaksi()\nlatih, uji = hari < 120, hari >= 120               # 4 bulan latih, 2 bulan uji  #<1>\nprint(f\"transaksi uji: {uji.sum()}, tidak wajar: {y[uji].sum()} ({y[uji].mean():.2%})\")\n\ndef laporan(nama, pred):\n    tn, fp, fn, tp = confusion_matrix(y[uji], pred, labels=[0, 1]).ravel()   #<2>\n    akurasi = (tp + tn) / (tp + tn + fp + fn)\n    presisi = tp / (tp + fp) if tp + fp else float(\"nan\")\n    recall = tp / (tp + fn)\n    f1 = 2 * presisi * recall / (presisi + recall) if tp else 0.0\n    print(f\"{nama:21s}{tp:4d}{fp:5d}{fn:4d} | {akurasi:.4f}  {presisi:7.3f} {recall:6.3f} {f1:5.3f}\")\n\nprint(f\"{'model':21s}  TP   FP  FN | akurasi presisi recall    F1\")\nlaporan(\"selalu 'wajar'\", np.zeros(uji.sum(), int))                    #<3>\nmodel = LogisticRegression(max_iter=1000).fit(X[latih], y[latih])\np = model.predict_proba(X[uji])[:, 1]\nlaporan(\"logistik, ambang 0.5\", (p > 0.5).astype(int))\nlaporan(\"logistik, ambang 0.1\", (p > 0.1).astype(int))                 #<4>\n",
"bab06/banding_model.py": "import numpy as np\nfrom scipy.stats import binomtest\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import average_precision_score\nfrom data_transaksi import buat_transaksi\n\nX, y, hari = buat_transaksi()\nlatih, uji = hari < 120, hari >= 120\npA = LogisticRegression(max_iter=1000).fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]\npB = HistGradientBoostingClassifier(random_state=0).fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]\nt = y[uji]\n\n# Uji McNemar: hanya pasangan yang berbeda pendapat yang informatif  #<1>\nbA, bB = (pA > 0.5) == t, (pB > 0.5) == t\nn01, n10 = np.sum(bA & ~bB), np.sum(~bA & bB)\nprint(f\"McNemar: A benar B salah = {n01}, A salah B benar = {n10}, \"\n      f\"p = {binomtest(n01, n01 + n10).pvalue:.3f}\")\n\n# Bootstrap selisih AP: ambil ulang transaksi uji dengan pengembalian  #<2>\nrng = np.random.default_rng(0)\nselisih = []\nfor _ in range(2000):\n    i = rng.integers(0, len(t), len(t))\n    if t[i].sum() == 0:\n        continue\n    selisih.append(average_precision_score(t[i], pA[i]) - average_precision_score(t[i], pB[i]))\nlo, hi = np.percentile(selisih, [2.5, 97.5])                          #<3>\nprint(f\"AP logistik - AP boosting = {average_precision_score(t, pA) - average_precision_score(t, pB):.3f}, \"\n      f\"selang bootstrap 95% [{lo:.3f}, {hi:.3f}]\")\n",
"bab06/data_transaksi.py": "\"\"\"Transaksi dompet digital sintetis (ilustrasi). Sekitar 1% tidak wajar.\n\nFitur: log nominal, jam (dalam bentuk sin/cos), jarak dari lokasi biasa (km,\nlog), merchant baru (0/1), banyak transaksi 24 jam terakhir, umur akun (log\nhari). Pola penipuan bergeser pelan dari bulan ke bulan (drift).\n\"\"\"\nimport numpy as np\n\nNAMA_FITUR = [\"log_nominal\", \"jam_sin\", \"jam_cos\", \"log_jarak\",\n              \"merchant_baru\", \"frek_24j\", \"log_umur_akun\"]\n\ndef buat_transaksi(n=60_000, laju=0.01, seed=0):\n    rng = np.random.default_rng(seed)\n    hari = np.sort(rng.integers(0, 180, n))            # enam bulan\n    y = (rng.random(n) < laju).astype(int)\n    geser = hari / 180                                 # drift: 0 -> 1\n    jam = np.where(y == 1, rng.normal(2 + 3 * geser, 3, n), rng.normal(14, 4, n)) % 24\n    X = np.column_stack([\n        np.where(y == 1, rng.normal(13.0 - 1.0 * geser, 1.2, n), rng.normal(11.0, 1.3, n)),\n        np.sin(2 * np.pi * jam / 24), np.cos(2 * np.pi * jam / 24),\n        np.where(y == 1, rng.normal(3.0 - 1.2 * geser, 1.3, n), rng.normal(1.0, 1.0, n)),\n        (rng.random(n) < np.where(y == 1, 0.6, 0.1)).astype(float),\n        rng.poisson(np.where(y == 1, 4 - 2 * geser, 2)),\n        np.where(y == 1, rng.normal(3.0, 1.5, n), rng.normal(5.5, 1.2, n)),\n    ])\n    return X, y, hari\n",
"bab06/kebocoran.py": "import numpy as np\nfrom sklearn.feature_selection import SelectKBest, f_classif\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.metrics import average_precision_score\nfrom data_transaksi import buat_transaksi\n\n# (a) Kebocoran lewat seleksi fitur: 2000 fitur derau murni, label acak  #<1>\nrng = np.random.default_rng(0)\nZ, t = rng.normal(size=(200, 2000)), rng.integers(0, 2, 200)\nbocor = SelectKBest(f_classif, k=20).fit(Z, t).transform(Z)      # pilih dulu, CV kemudian\nauc_bocor = cross_val_score(LogisticRegression(), bocor, t, cv=5, scoring=\"roc_auc\")\nbenar = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())   #<2>\nauc_benar = cross_val_score(benar, Z, t, cv=5, scoring=\"roc_auc\")\nprint(f\"label acak | AUC CV bocor = {auc_bocor.mean():.2f} | AUC CV benar = {auc_benar.mean():.2f}\")\n\n# (b) Panah waktu: CV acak vs CV berurutan, dibanding kinerja masa depan  #<3>\nX, y, hari = buat_transaksi()\nlatih, uji = hari < 120, hari >= 120\nm = LogisticRegression(max_iter=1000)\nfor nama, cv in [(\"KFold acak\", KFold(5, shuffle=True, random_state=0)),\n                 (\"TimeSeriesSplit\", TimeSeriesSplit(5))]:\n    s = cross_val_score(m, X[latih], y[latih], cv=cv, scoring=\"average_precision\")\n    print(f\"{nama:16s}: AP taksiran CV = {s.mean():.3f}\")\np = m.fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]\nprint(f\"{'masa depan':16s}: AP data uji  = {average_precision_score(y[uji], p):.3f}\")\n",
"bab06/koreksi_prior.py": "import numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom data_transaksi import buat_transaksi\n\nX, y, hari = buat_transaksi()\nlatih, uji = hari < 120, hari >= 120\nrng = np.random.default_rng(0)\npos = np.where(latih & (y == 1))[0]\nneg = rng.choice(np.where(latih & (y == 0))[0], size=len(pos), replace=False)\nseimbang = np.r_[pos, neg]                          # undersampling 50:50  #<1>\np = LogisticRegression(max_iter=1000).fit(X[seimbang], y[seimbang]).predict_proba(X[uji])[:, 1]\n\npi_latih, pi_baru = 0.5, y[latih].mean()            # prior saat latih vs nyata\nr = p * pi_baru / pi_latih                          # koreksi prior (teorema Bayes)  #<2>\np_kor = r / (r + (1 - p) * (1 - pi_baru) / (1 - pi_latih))\n\ndef ece(p, t, k=10):                                # galat kalibrasi harapan  #<3>\n    kelompok = np.minimum((p * k).astype(int), k - 1)\n    return sum(np.abs(p[kelompok == b].mean() - t[kelompok == b].mean()) * np.mean(kelompok == b)\n               for b in range(k) if np.any(kelompok == b))\n\nprint(f\"prevalensi nyata di data uji: {y[uji].mean():.4f}\")\nfor nama, q in [(\"tanpa koreksi\", p), (\"dengan koreksi\", p_kor)]:\n    ll = -np.mean(y[uji] * np.log(q) + (1 - y[uji]) * np.log(1 - q))\n    print(f\"{nama:14s}: rerata p {q.mean():.4f} | log-loss {ll:.4f} | \"\n          f\"ECE {ece(q, y[uji]):.4f} | p>0.5: {np.sum(q > 0.5)}\")\n",
"bab06/roc_biaya.py": "import numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import roc_auc_score, average_precision_score\nfrom data_transaksi import buat_transaksi\n\nX, y, hari = buat_transaksi()\nlatih, uji = hari < 120, hari >= 120\nmodel = {\"logistik\": LogisticRegression(max_iter=1000),\n         \"gradient boosting\": HistGradientBoostingClassifier(random_state=0)}  #<1>\np = {k: m.fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1] for k, m in model.items()}\nfor k, pk in p.items():\n    print(f\"{k:18s} ROC-AUC {roc_auc_score(y[uji], pk):.3f} | \"\n          f\"AP (luas PR) {average_precision_score(y[uji], pk):.3f}\")       #<2>\n\nc_fp, c_fn = 20_000, 2_000_000        # biaya periksa manual vs kerugian lolos  #<3>\nambang_bayes = c_fp / (c_fp + c_fn)\nfor ambang in (0.5, ambang_bayes):\n    pred = p[\"logistik\"] > ambang\n    fp = np.sum(pred & (y[uji] == 0)); fn = np.sum(~pred & (y[uji] == 1))\n    print(f\"ambang {ambang:.4f}: FP={fp:4d} FN={fn:3d} | \"\n          f\"biaya total Rp{(fp * c_fp + fn * c_fn) / 1e6:6.1f} juta\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab06')
print('folder kerja:', os.getcwd())

## Listing 6.1: Akurasi, presisi, recall, dan F1 pada data tidak seimbang

Berkas: `kode/bab06/akurasi_menipu.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from data_transaksi import buat_transaksi

X, y, hari = buat_transaksi()
latih, uji = hari < 120, hari >= 120               # 4 bulan latih, 2 bulan uji  # (1)
print(f"transaksi uji: {uji.sum()}, tidak wajar: {y[uji].sum()} ({y[uji].mean():.2%})")

def laporan(nama, pred):
    tn, fp, fn, tp = confusion_matrix(y[uji], pred, labels=[0, 1]).ravel()   # (2)
    akurasi = (tp + tn) / (tp + tn + fp + fn)
    presisi = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn)
    f1 = 2 * presisi * recall / (presisi + recall) if tp else 0.0
    print(f"{nama:21s}{tp:4d}{fp:5d}{fn:4d} | {akurasi:.4f}  {presisi:7.3f} {recall:6.3f} {f1:5.3f}")

print(f"{'model':21s}  TP   FP  FN | akurasi presisi recall    F1")
laporan("selalu 'wajar'", np.zeros(uji.sum(), int))                    # (3)
model = LogisticRegression(max_iter=1000).fit(X[latih], y[latih])
p = model.predict_proba(X[uji])[:, 1]
laporan("logistik, ambang 0.5", (p > 0.5).astype(int))
laporan("logistik, ambang 0.1", (p > 0.1).astype(int))                 # (4)

## Listing 6.2: Dua bentuk kebocoran: seleksi fitur dan panah waktu

Berkas: `kode/bab06/kebocoran.py`

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score
from data_transaksi import buat_transaksi

# (a) Kebocoran lewat seleksi fitur: 2000 fitur derau murni, label acak  # (1)
rng = np.random.default_rng(0)
Z, t = rng.normal(size=(200, 2000)), rng.integers(0, 2, 200)
bocor = SelectKBest(f_classif, k=20).fit(Z, t).transform(Z)      # pilih dulu, CV kemudian
auc_bocor = cross_val_score(LogisticRegression(), bocor, t, cv=5, scoring="roc_auc")
benar = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())   # (2)
auc_benar = cross_val_score(benar, Z, t, cv=5, scoring="roc_auc")
print(f"label acak | AUC CV bocor = {auc_bocor.mean():.2f} | AUC CV benar = {auc_benar.mean():.2f}")

# (b) Panah waktu: CV acak vs CV berurutan, dibanding kinerja masa depan  # (3)
X, y, hari = buat_transaksi()
latih, uji = hari < 120, hari >= 120
m = LogisticRegression(max_iter=1000)
for nama, cv in [("KFold acak", KFold(5, shuffle=True, random_state=0)),
                 ("TimeSeriesSplit", TimeSeriesSplit(5))]:
    s = cross_val_score(m, X[latih], y[latih], cv=cv, scoring="average_precision")
    print(f"{nama:16s}: AP taksiran CV = {s.mean():.3f}")
p = m.fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]
print(f"{'masa depan':16s}: AP data uji  = {average_precision_score(y[uji], p):.3f}")

## Listing 6.3: ROC, presisi–recall, dan ambang berbasis biaya

Berkas: `kode/bab06/roc_biaya.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from data_transaksi import buat_transaksi

X, y, hari = buat_transaksi()
latih, uji = hari < 120, hari >= 120
model = {"logistik": LogisticRegression(max_iter=1000),
         "gradient boosting": HistGradientBoostingClassifier(random_state=0)}  # (1)
p = {k: m.fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1] for k, m in model.items()}
for k, pk in p.items():
    print(f"{k:18s} ROC-AUC {roc_auc_score(y[uji], pk):.3f} | "
          f"AP (luas PR) {average_precision_score(y[uji], pk):.3f}")       # (2)

c_fp, c_fn = 20_000, 2_000_000        # biaya periksa manual vs kerugian lolos  # (3)
ambang_bayes = c_fp / (c_fp + c_fn)
for ambang in (0.5, ambang_bayes):
    pred = p["logistik"] > ambang
    fp = np.sum(pred & (y[uji] == 0)); fn = np.sum(~pred & (y[uji] == 1))
    print(f"ambang {ambang:.4f}: FP={fp:4d} FN={fn:3d} | "
          f"biaya total Rp{(fp * c_fp + fn * c_fn) / 1e6:6.1f} juta")

## Listing 6.4: Undersampling dan koreksi prior

Berkas: `kode/bab06/koreksi_prior.py`

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from data_transaksi import buat_transaksi

X, y, hari = buat_transaksi()
latih, uji = hari < 120, hari >= 120
rng = np.random.default_rng(0)
pos = np.where(latih & (y == 1))[0]
neg = rng.choice(np.where(latih & (y == 0))[0], size=len(pos), replace=False)
seimbang = np.r_[pos, neg]                          # undersampling 50:50  # (1)
p = LogisticRegression(max_iter=1000).fit(X[seimbang], y[seimbang]).predict_proba(X[uji])[:, 1]

pi_latih, pi_baru = 0.5, y[latih].mean()            # prior saat latih vs nyata
r = p * pi_baru / pi_latih                          # koreksi prior (teorema Bayes)  # (2)
p_kor = r / (r + (1 - p) * (1 - pi_baru) / (1 - pi_latih))

def ece(p, t, k=10):                                # galat kalibrasi harapan  # (3)
    kelompok = np.minimum((p * k).astype(int), k - 1)
    return sum(np.abs(p[kelompok == b].mean() - t[kelompok == b].mean()) * np.mean(kelompok == b)
               for b in range(k) if np.any(kelompok == b))

print(f"prevalensi nyata di data uji: {y[uji].mean():.4f}")
for nama, q in [("tanpa koreksi", p), ("dengan koreksi", p_kor)]:
    ll = -np.mean(y[uji] * np.log(q) + (1 - y[uji]) * np.log(1 - q))
    print(f"{nama:14s}: rerata p {q.mean():.4f} | log-loss {ll:.4f} | "
          f"ECE {ece(q, y[uji]):.4f} | p>0.5: {np.sum(q > 0.5)}")

## Listing 6.5: Uji McNemar dan selang bootstrap untuk selisih kinerja

Berkas: `kode/bab06/banding_model.py`

In [ ]:
import numpy as np
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from data_transaksi import buat_transaksi

X, y, hari = buat_transaksi()
latih, uji = hari < 120, hari >= 120
pA = LogisticRegression(max_iter=1000).fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]
pB = HistGradientBoostingClassifier(random_state=0).fit(X[latih], y[latih]).predict_proba(X[uji])[:, 1]
t = y[uji]

# Uji McNemar: hanya pasangan yang berbeda pendapat yang informatif  # (1)
bA, bB = (pA > 0.5) == t, (pB > 0.5) == t
n01, n10 = np.sum(bA & ~bB), np.sum(~bA & bB)
print(f"McNemar: A benar B salah = {n01}, A salah B benar = {n10}, "
      f"p = {binomtest(n01, n01 + n10).pvalue:.3f}")

# Bootstrap selisih AP: ambil ulang transaksi uji dengan pengembalian  # (2)
rng = np.random.default_rng(0)
selisih = []
for _ in range(2000):
    i = rng.integers(0, len(t), len(t))
    if t[i].sum() == 0:
        continue
    selisih.append(average_precision_score(t[i], pA[i]) - average_precision_score(t[i], pB[i]))
lo, hi = np.percentile(selisih, [2.5, 97.5])                          # (3)
print(f"AP logistik - AP boosting = {average_precision_score(t, pA) - average_precision_score(t, pB):.3f}, "
      f"selang bootstrap 95% [{lo:.3f}, {hi:.3f}]")